In [13]:
import os
import pandas as pd
import pdfreader
import logging as logger
import datetime
import json
import math as mt
import camelot

In [ ]:
def process_pdf_auto(
                        filename: str,
                        output_path,
                        extract_images: bool = False,
                        filecontent=None,
                        direction="dataframe",
                        save_text_coordinates_to_local: bool = False,
                        save_file_to_local: bool = False,
                        extract_by_page: bool = True):

    try:
        pdf_reader = PDFReader(
            file_path=filename,
            output_path=output_path,
            extract_images=extract_images
        )
        if filecontent:
            data_coordinates, doc_total_pages = pdf_reader.read_pdf_file(filecontent=filecontent)
        else:
            data_coordinates, doc_total_pages = pdf_reader.read_pdf_file()


        # SAVE data coordinates is useful to investigate a problem or understand how data is structured within a document
        if save_text_coordinates_to_local:
            data_coordinates_filename = f'data_coordinates_{filename.split(".")[0]}_{str(datetime.date.today())}.csv'
            pd.DataFrame(data_coordinates).to_csv(
                os.path.join(output_path, data_coordinates_filename)
            )

        df_pdf_data = pd.DataFrame(data_coordinates)



        # Extract data as tables
        if direction=="dataframe":
            procesed_file_content = extract_auto_tables(df_pdf_data=df_pdf_data,
                                                                filename=filename,
                                                                extract_by_page=extract_by_page)
        # Extract data as key/value
        else:
            procesed_file_content = extract_auto_keyvalues(df_pdf_data=df_pdf_data,
                                                                filename=filename,
                                                                extract_by_page=extract_by_page)
        if save_file_to_local:
            json_filename = F"{filename.split('.')[0]}.json"
            with open(os.path.join(output_path, json_filename), "w") as f:
                json.dump(procesed_file_content, f)

        return procesed_file_content
    except Exception as e:
        print(
            f"Error occurred during process_pdf_auto(filename={filename})"
        )
        print(e)
        return None

def extract_auto_keyvalues( df_pdf_data, filename, max_treshold:int = 3, extract_by_page: bool = True):

    """
    :param df_pdf_data:
    :param filename:
    :param extract_by_page:
    :return:
    """


    max_pages = set(df_pdf_data.page_num)

    new_res = pd.DataFrame(columns=['page_num', 'text', 'bbox.x0', 'bbox.y0', 'bbox.x1', 'bbox.y1',
                                    'fontname', 'fontsize'
                                    ]
                            )
    for current_page in max_pages:
        temp_string = ""
        fontname_list = []
        fontsize_list = []
        res_list = df_pdf_data[df_pdf_data.page_num == current_page]
        res_list = res_list.sort_values(by=["bbox.y0", "bbox.x0"], ascending=[False, True], axis=0)

        res_list["prev_bbox.y0"] = res_list["bbox.y0"].shift()
        res_list["prev_bbox.x0"] = res_list["bbox.x0"].shift()
        res_list["prev_fontname"] = res_list["fontname"].shift()
        res_list["prev_fontsize"] = res_list["fontsize"].shift()
        res_list["diff_bbox.x0"] = res_list["bbox.x0"] - res_list["prev_bbox.x0"]
        res_list["diff_bbox.x0"] = res_list.apply(
            lambda x: 1000 if x["bbox.y0"] != x["prev_bbox.y0"] else x["diff_bbox.x0"], axis=1)
        res_list["ratio_fontsize_diff_x0"] = res_list["diff_bbox.x0"] / res_list["fontsize"]

        for i0, (_, r) in enumerate(res_list.iterrows()):
            # Clean text
            temp_text = r["text"].replace(u'\xa0', u' ')

            # Check if is same content
            if r["fontname"] == r["prev_fontname"] and r["fontsize"] == r["prev_fontsize"] and (
                    np.isnan(r["ratio_fontsize_diff_x0"]) == True or r["ratio_fontsize_diff_x0"] < max_treshold):
                temp_string += temp_text
                if temp_string == "":
                    page_num = r["page_num"]
                    bbox_x0 = r["bbox.x0"]
                    bbox_y0 = r["bbox.y0"]

                if temp_text != "":
                    fontname_list.append(r['fontname'])
                    fontsize_list.append(r['fontsize'])

                bbox_x1 = r["bbox.x1"]
                bbox_y1 = r["bbox.y1"]

            elif temp_string != "":
                fontname = max(set(fontname_list), key=fontname_list.count)
                fontsize = max(set(fontsize_list), key=fontsize_list.count)

                new_res = new_res.append({"page_num": page_num,
                                            "text": temp_string.strip(),
                                            "bbox.x0": bbox_x0,
                                            "bbox.y0": bbox_y0,
                                            "bbox.x1": bbox_x1,
                                            "bbox.y1": bbox_y1,
                                            "fontname": fontname,
                                            "fontsize": fontsize, }, ignore_index=True)

                temp_string = temp_text
                page_num = r["page_num"]
                bbox_x0 = r["bbox.x0"]
                bbox_y0 = r["bbox.y0"]
                fontname_list = []
                fontsize_list = []
                if temp_text != "":
                    fontname_list.append(r['fontname'])
                    fontsize_list.append(r['fontsize'])
                bbox_x1 = r["bbox.x1"]
                bbox_y1 = r["bbox.y1"]
            else:
                temp_string = temp_text
                page_num = r["page_num"]
                bbox_x0 = r["bbox.x0"]
                bbox_y0 = r["bbox.y0"]
                if temp_text != "":
                    fontname_list.append(r['fontname'])
                    fontsize_list.append(r['fontsize'])
                bbox_x1 = r["bbox.x1"]
                bbox_y1 = r["bbox.y1"]

            # Add last row
            if temp_string != "" and i0 == (len(res_list) - 1):
                fontname = max(set(fontname_list), key=fontname_list.count)
                fontsize = max(set(fontsize_list), key=fontsize_list.count)

                new_res = new_res.append({"page_num": page_num,
                                            "text": temp_string.strip(),
                                            "bbox.x0": bbox_x0,
                                            "bbox.y0": bbox_y0,
                                            "bbox.x1": bbox_x1,
                                            "bbox.y1": bbox_y1,
                                            "fontname": fontname,
                                            "fontsize": fontsize, }, ignore_index=True)

    page_list = []
    new_res = new_res.fillna("")
    for page_id in max_pages:
        key = None
        value = None
        fontname=None
        fontsize=None
        is_used = 0
        unused = []
        separator = ":"
        row_list = []
        row_data = []
        current_bbox_y0 = None
        prev_bbox_y0 = None
        row_id = 1

        for i0, (r_id, row) in enumerate(new_res[new_res.page_num == page_id].iterrows()):
            is_used = 0
            current_bbox_y0 = row["bbox.y0"]
            # Apprend row data to row list
            # print(F"test:{row.text},prev_y0:{prev_bbox_y0}, curr_y0:{current_bbox_y0}")
            if row_data and prev_bbox_y0 and current_bbox_y0 != prev_bbox_y0:
                if key:
                    row_data.append({key: "","fontname":fontname,"fontsize":fontsize})
                row_list.append({"row_id": row_id, "row_data": row_data})
                row_data = []
                row_id += 1
            prev_bbox_y0 = current_bbox_y0

            if row.text.find(separator) >= 0 and row.fontname.find("Bold") >= 0 and row.text != "":
                key = row.text
                fontname = row.fontname
                fontsize = row.fontsize
                is_used = 1
            elif key:
                value = row.text
                fontname = row.fontname
                fontsize = row.fontsize
                is_used = 1

            if is_used == 0 and row.text != "":
                if key:
                    row_data.append({"text": key,"fontname":fontname,"fontsize":fontsize})
                row_data.append({"text": row.text,"fontname":row.fontname,"fontsize":row.fontsize})
                key = None
                value = None
                fontname = None
                fontsize = None

            # Append key:value to row_data
            if key and (value or value == ""):
                row_data.append({key: value,"fontname":fontname,"fontsize":fontsize})
                key = None
                value = None
                fontname = None
                fontsize = None

        if (prev_bbox_y0 and current_bbox_y0 != prev_bbox_y0) or i0 == (len(new_res) - 1):
            if key:
                row_data.append({key: "","fontname":fontname,"fontsize":fontsize})
            if row_data:
                row_list.append({"row_id": row_id, "row_data": row_data})
            row_data = []
            row_id += 1

        # Append row_list to page data
        page_list.append({"page_id": page_id, "page_data": row_list})

    return page_list

def extract_auto_tables(df_pdf_data,filename,extract_by_page: bool = True):
        # group text that is in the same line and count number of element in each group
        df_pdf_data["bbox.y0_rounded"] = df_pdf_data["bbox.y0"].apply(lambda x: mt.floor(x))
        df_pdf_data_group = df_pdf_data.groupby(["page_num", "bbox.y0_rounded"])
        df_pdf_data_group_count = df_pdf_data_group["bbox.x0"].count().rename("count_row_cols").reset_index()
        df_pdf_data_group_count = df_pdf_data_group_count.sort_values(by=["page_num", "bbox.y0_rounded"],
                                                                        ascending=[True,False])
        # Max columns per page
        df_pdf_data_group_page = df_pdf_data_group_count.groupby(["page_num"])
        df_pdf_data_group_count_page = df_pdf_data_group_page["count_row_cols"].max().rename("max_cols_per_page").reset_index()
        df_pdf_data_group_count_page = df_pdf_data_group_count_page.sort_values(by=["page_num"], ascending=[True])

        # Merge with main data
        df_pdf_data_merge = pd.merge(df_pdf_data_group_count,
                                        df_pdf_data,
                                        on=["page_num", "bbox.y0_rounded"],
                                        how="left")

        df_pdf_data_merge = pd.merge(df_pdf_data_group_count_page,
                                        df_pdf_data_merge,
                                        on=["page_num"],
                                        how="left")

        df_pdf_data_merge = df_pdf_data_merge.sort_values(by=["page_num", "bbox.y0", "bbox.x0"],
                                                            ascending=[True, False, True])

        #Define group of data
        df_pdf_data_merge["is_table"] = df_pdf_data_merge.count_row_cols.apply(lambda x: 1 if x>1 else 0)
        df_pdf_data_merge["group_id"] = 0
        df_pdf_data_merge["direction"] = "dataframe"

        if extract_by_page:
            df_pdf_data_merge_count_by_page = (df_pdf_data_merge
                                                .groupby("page_num")
                                                .agg({"is_table":"sum",
                                                        "text": "count"})
                                                .reset_index()
                                                )
            df_pdf_data_merge_count_by_page.columns = ["page_num","count_tables","count_all"]
            df_pdf_data_merge_count_by_page["table_ratio"] = (df_pdf_data_merge_count_by_page.count_tables/df_pdf_data_merge_count_by_page.count_all)
            df_pdf_data_merge = pd.merge(df_pdf_data_merge_count_by_page,
                                            df_pdf_data_merge,
                                            on=["page_num"],
                                            how="left")

            TABLE_TRESHOLD = 0.5
            df_pdf_data_merge["direction"] = df_pdf_data_merge.table_ratio.apply(lambda x: "dataframe" if x>TABLE_TRESHOLD else "text")


        else:
            group_id=0
            prev_row = None
            max_row = len(df_pdf_data_merge)

            #Shift prev and next
            df_pdf_data_merge["prev_1_page_num"]=df_pdf_data_merge.page_num.shift(1)
            df_pdf_data_merge["prev_1_is_table"]=df_pdf_data_merge.is_table.shift(1)
            df_pdf_data_merge["next_1_page_num"]=df_pdf_data_merge.page_num.shift(-1)
            df_pdf_data_merge["next_1_is_table"]=df_pdf_data_merge.is_table.shift(-1)
            df_pdf_data_merge["next_2_page_num"]=df_pdf_data_merge.page_num.shift(-2)
            df_pdf_data_merge["next_2_is_table"]=df_pdf_data_merge.is_table.shift(-2)

            current_is_table = None
            for i, row in df_pdf_data_merge.iterrows():
                if i != 0:
                    #Compare current row with previous row and with 2 next rows:
                    # If text is embeded within a dataframe then include text to dataframe
                    if (row["prev_1_page_num"] == row["page_num"]
                        and current_is_table == row["is_table"]
                    ) or (row["prev_1_page_num"] == row["next_1_page_num"]
                            and current_is_table == row["next_1_is_table"]
                    )or (row["prev_1_page_num"] == row["next_2_page_num"]
                            and current_is_table == row["next_2_is_table"]

                    ):
                        df_pdf_data_merge.at[i, "group_id"] = group_id
                        df_pdf_data_merge.at[i, "is_table"] = current_is_table
                        df_pdf_data_merge.at[i+1, "prev_1_is_table"] = current_is_table
                    else:
                        current_is_table = row["is_table"]
                        group_id += 1
                        df_pdf_data_merge.at[i, "group_id"] = group_id

            df_pdf_data_merge["direction"] = df_pdf_data_merge.is_table.apply(lambda x: "text" if x==0 else "dataframe")

        df_pdf_data_merge["section_name"] = df_pdf_data_merge.apply(lambda x: f"Page-{x.page_num}_group-{x.group_id}",
                                                                    axis=1)


        df_pdf_data_merge_group = df_pdf_data_merge.groupby(["page_num",
                                                                "section_name",
                                                                "direction"
                                                                ]).agg({"bbox.x0": "min",
                                                                        "bbox.x1": "max",
                                                                        "bbox.y0": "max",
                                                                        "bbox.y1": "min"}).reset_index()

        # PROCESS file content
        list_data = []
        table_areas = ["0,755,755,0"]
        for i, row in df_pdf_data_merge_group.iterrows():
            if extract_by_page == False:
                table_areas = [f"{row['bbox.x0']-10},{row['bbox.y0']+10},{row['bbox.x1']+10},{row['bbox.y1']-10}"]
            tables = camelot.read_pdf(filename,
                pages=f"{row['page_num']}",
                flavor="stream",
                table_areas=table_areas,
                strip_text="\n",
                flag_size=True,
                split_text=False,
                col_tol=100,
                suppress_stdout=True
            )
            #Concatenate result to previous subpages: This manage sections that cross multiple pages
            temp_df = tables[0].df
            if not temp_df.empty:
                data_content = temp_df.to_dict(orient="records")
                if row["direction"] == "text":
                    data_content = json.dumps(data_content)#concat_list_to_string(data_content)

                list_data.append({"page_num": row["page_num"],
                                    "section_id": i,
                                    "section_name": row["section_name"],
                                    "direction": row["direction"],
                                    "data_list": data_content
                                    }
                                    )


        # Add processed report content to result list
        procesed_file_content = {
            "source_filename": filename,
            "process_date": str(datetime.date.today()),
            "total_pages": str(df_pdf_data_merge.page_num.max()),
            "report_type": "Auto-Extraction",
            "report_content": list_data,
        }

        return procesed_file_content

In [7]:
process_pdf_auto(filename='./in/2021-111P.pdf',
                 output_path='./out',
                 extract_images=False,
                 filecontent=None,
                 direction="dataframe",
                 save_text_coordinates_to_local= False,
                 save_file_to_local=False,
                 extract_by_page=True)

Error occurred during process_pdf_auto(filename=./in/2021-111P.pdf)
'module' object is not callable


In [11]:
filename='./in/2021-111P.pdf',
output_path='./out',
extract_images=False,
filecontent=None,
direction="dataframe",
save_text_coordinates_to_local= False,
save_file_to_local=False,
extract_by_page=True

pdf_reader = PDFReader(
    file_path=filename,
    output_path=output_path,
    extract_images=extract_images
)
if filecontent:
    data_coordinates, doc_total_pages = pdf_reader.read_pdf_file(filecontent=filecontent)
else:
    data_coordinates, doc_total_pages = pdf_reader.read_pdf_file()


# SAVE data coordinates is useful to investigate a problem or understand how data is structured within a document
if save_text_coordinates_to_local:
    data_coordinates_filename = f'data_coordinates_{filename.split(".")[0]}_{str(datetime.date.today())}.csv'
    pd.DataFrame(data_coordinates).to_csv(
        os.path.join(output_path, data_coordinates_filename)
    )

df_pdf_data = pd.DataFrame(data_coordinates)



# Extract data as tables
if direction=="dataframe":
    procesed_file_content = extract_auto_tables(df_pdf_data=df_pdf_data,
                                                        filename=filename,
                                                        extract_by_page=extract_by_page)
# Extract data as key/value
else:
    procesed_file_content = extract_auto_keyvalues(df_pdf_data=df_pdf_data,
                                                        filename=filename,
                                                        extract_by_page=extract_by_page)
if save_file_to_local:
    json_filename = F"{filename.split('.')[0]}.json"
    with open(os.path.join(output_path, json_filename), "w") as f:
        json.dump(procesed_file_content, f)

AttributeError: module 'pdfreader' has no attribute 'PDFReader'

In [3]:
import sys
import os
import io
import glob
import json
import html 
#from PyPDF2 import PdfReader
#from markdownify import markdownify as md
from azure.ai.formrecognizer import DocumentAnalysisClient

from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest, ContentFormat, AnalyzeResult

from azure.core.credentials import AzureKeyCredential
#from azure.identity import AzureDeveloperCliCredential
from azure.storage.blob import BlobServiceClient
import requests
from dotenv import load_dotenv
load_dotenv("../ez_env.env")

True

In [5]:

def get_document_markdown(url_source=None,bytes_source=None):
    document_intelligence_creds = AzureKeyCredential(os.getenv("AZURE_FORM_RECOGNIZER_KEY"))

    # Document Intelligence Output to markdown format
    document_intelligence_client = DocumentIntelligenceClient(endpoint=os.getenv("AZURE_FORM_RECOGNIZER_ENDPOINT"), 
                                                                credential=document_intelligence_creds,
                                                                #api_version="2023-07-31"
                                                                )
    result = None
    if url_source:
        poller = document_intelligence_client.begin_analyze_document(
            model_id="prebuilt-layout",
            analyze_request=AnalyzeDocumentRequest(url_source=url_source),
            output_content_format=ContentFormat.MARKDOWN,
        )
        result = poller.result()
    if bytes_source:
        with open(bytes_source, "rb") as f:
            poller = document_intelligence_client.begin_analyze_document(
                model_id="prebuilt-layout", 
                analyze_request=f, 
                content_type="application/octet-stream",
                output_content_format=ContentFormat.MARKDOWN,
            )
        result = poller.result()

    if result:
        return result
    else:
        return None

In [ ]:
res=get_document_markdown(bytes_source='../in/WSP prop. HQP_Analyse_de_risque_2021-Ponts_roulants_signé.pdf')

In [1]:
from pdfminer.high_level import extract_pages
from pdfminer.layout import LTTextContainer, LTChar,LTLine,LAParams
import os
path=r'../in/2021-111P.pdf'

Extract_Data=[]
stop=False
for page_layout in extract_pages(path):
    if stop==True:
        break
    for element in page_layout:
        if isinstance(element, LTTextContainer):
            stop=True
            break
            for text_line in element:
                try:
                    for character in text_line:
                        if isinstance(character, LTChar):
                            Font_size=character.size
                except:
                    Font_size=0
            Extract_Data.append([Font_size,(element.get_text())])

In [ ]:
import os
import sys

# REf: https://pdfminersix.readthedocs.io/en/latest/reference/commandline.html
# Ref: https://readthedocs.org/projects/pdfminer-docs/downloads/pdf/latest/
# execute command
# py C:\Users\ezzatdemnati\venvs\venv311\Scripts\pdf2txt.py -o=out/output2.html -O out/img  "in/P-1223-23 - Final.pdf"
os.system(r'')


1

: 

In [ ]:
search_filter = f"search.in(name, '{','.join(doc)}', ',')"
print(F"search_filter: {search_filter}")
results = search_client.search(
    #query_type="Hybrid",
    #query_language="en-us",
    semantic_configuration_name="default",
    search_text=question,
    vector_queries=[VectorizedQuery(vector=generate_embeddings(question), k_nearest_neighbors=3, fields="vector")],
    filter=search_filter,
    #select=["recipe_id", "recipe", "recipe_category", "recipe_name", "description"],
    top=6,
)

In [ ]:
#########################################################################

' This is what we are trying to do:\n1) Transfer information from PDF file to PDF document object. This is done using parser\n2) Open the PDF file\n3) Parse the file using PDFParser object\n4) Assign the parsed content to PDFDocument object\n5) Now the information in this PDFDocumet object has to be processed. For this we need\n   PDFPageInterpreter, PDFDevice and PDFResourceManager\n 6) Finally process the file page by page \n'

In [1]:
import os
from pdfminer.pdfparser import PDFParser
from pdfminer.pdfdocument import PDFDocument
from pdfminer.pdfpage import PDFPage
# From PDFInterpreter import both PDFResourceManager and PDFPageInterpreter
from pdfminer.pdfinterp import PDFResourceManager, PDFPageInterpreter
from pdfminer.pdfdevice import PDFDevice
# Import this to raise exception whenever text extraction from PDF is not allowed
from pdfminer.pdfpage import PDFTextExtractionNotAllowed
from pdfminer.layout import LAParams, LTTextBox, LTTextLine, LTRect, LTTextLineHorizontal 
from pdfminer.converter import PDFPageAggregator

''' This is what we are trying to do:
1) Transfer information from PDF file to PDF document object. This is done using parser
2) Open the PDF file
3) Parse the file using PDFParser object
4) Assign the parsed content to PDFDocument object
5) Now the information in this PDFDocumet object has to be processed. For this we need
   PDFPageInterpreter, PDFDevice and PDFResourceManager
 6) Finally process the file page by page 
'''



def read_pdf_file(filename):
	
	#log_file = os.path.join(base_path + "out/" + "pdf_log.txt")

	password = ""

	# Open and read the pdf file in binary mode
	fp = open(filename, "rb")

	# Create parser object to parse the pdf content
	parser = PDFParser(fp)

	# Store the parsed content in PDFDocument object
	document = PDFDocument(parser, password)

	# Check if document is extractable, if not abort
	if not document.is_extractable:
		raise PDFTextExtractionNotAllowed
		
	# Create PDFResourceManager object that stores shared resources such as fonts or images
	rsrcmgr = PDFResourceManager()

	# set parameters for analysis
	laparams = LAParams()

	# Create a PDFDevice object which translates interpreted information into desired format
	# Device needs to be connected to resource manager to store shared resources
	# device = PDFDevice(rsrcmgr)
	# Extract the decive to page aggregator to get LT object elements
	device = PDFPageAggregator(rsrcmgr, laparams=laparams)

	# Create interpreter object to process page content from PDFDocument
	# Interpreter needs to be connected to resource manager for shared resources and device 
	interpreter = PDFPageInterpreter(rsrcmgr, device)

	doc_pages = PDFPage.create_pages(document)

	doc_pages = PDFPage.create_pages(document)
	# Ok now that we have everything to process a pdf document, lets process it page by page
	i=0
	stop=0
	extracted_text = []
	for page in doc_pages:
		i+=1
		# As the interpreter processes the page stored in PDFDocument object
		interpreter.process_page(page)
		# The device renders the layout from interpreter
		layout = device.get_result()
		# Out of the many LT objects within layout, we are interested in LTTextBox and LTTextLine
		for lt_obj in layout:
			if (isinstance(lt_obj, LTTextBox) 
				or isinstance(lt_obj, LTTextLine) 
				or isinstance(lt_obj, LTRect)
			):
				def get_fontname(lt_obj):
					try:
						return lt_obj.fontname
					except:
						return None
				def get_size(lt_obj):
					try:
						return lt_obj.size
					except:
						return None
				def get_direction(lt_obj):
					try:
						return lt_obj.direction
					except:
						return None
					
				try:
					extracted_text.append({"text":lt_obj.get_text(),
											"bbox":lt_obj.bbox,
											"fontname":get_fontname(lt_obj),
											"fontsize":get_size(lt_obj),
											"page_num":i,
											"type":type(lt_obj),
											"direction":get_direction(lt_obj),
											"bbox.x0":lt_obj.bbox[0],
											"bbox.y0":lt_obj.bbox[1],
											"bbox.x1":lt_obj.bbox[2],
											"bbox.y1":lt_obj.bbox[3]})
				except:
					if isinstance(lt_obj, LTRect):
						#print(f"pass:{lt_obj}")
						pass
					else:
						print(f"RAISE:{lt_obj}")
						raise

			else:
				print(f"page:{i}")
				print(lt_obj)
				if isinstance(lt_obj, LTRect) :
					stop+=1
			if stop>1:
				break	
		if stop==True:
			break	
	#close the pdf file
	fp.close()

	return extracted_text

In [5]:
base_path = "../"
my_file = os.path.join(base_path + "in/" + "2021-111P.pdf")

extracted_text = read_pdf_file(my_file)
extracted_text

page:1
<LTFigure(Im0) 57.660,697.390,150.660,747.440 matrix=[93.00,0.00,0.00,50.05, (57.66,697.39)]>
page:16
<LTFigure(Im0) 161.750,121.381,450.250,382.381 matrix=[288.50,0.00,0.00,261.00, (161.75,121.38)]>
page:17
<LTFigure(Im0) 159.000,382.306,453.171,679.306 matrix=[294.17,0.00,0.00,297.00, (159.00,382.31)]>
page:20
<LTFigure(Im0) 90.000,352.817,522.000,534.817 matrix=[432.00,0.00,0.00,182.00, (90.00,352.82)]>
page:22
<LTFigure(Im0) 140.025,335.796,507.975,558.266 matrix=[367.95,0.00,0.00,222.47, (140.02,335.80)]>
page:34
<LTFigure(Im0) 90.000,240.577,522.000,421.577 matrix=[432.00,0.00,0.00,181.00, (90.00,240.58)]>
page:85
<LTFigure(7eeb16a0-1383-48e6-9d8b-3af2c85a3dab) 0.000,0.000,612.000,792.000 matrix=[1.00,0.00,0.00,1.00, (0.00,0.00)]>
page:86
<LTFigure(5c909158-2445-4eff-a099-a78dd588003d) 0.000,0.000,612.000,792.000 matrix=[1.00,0.00,0.00,1.00, (0.00,0.00)]>
page:87
<LTFigure(3de30603-29e9-4410-bec3-daf93f518509) 0.000,0.000,612.000,792.000 matrix=[1.00,0.00,0.00,1.00, (0.00,

[{'text': 'Request for Proposal \n',
  'bbox': (231.84, 677.64, 387.69, 692.64),
  'fontname': None,
  'fontsize': None,
  'page_num': 1,
  'type': pdfminer.layout.LTTextBoxHorizontal,
  'direction': None,
  'bbox.x0': 231.84,
  'bbox.y0': 677.64,
  'bbox.x1': 387.69,
  'bbox.y1': 692.64},
 {'text': 'Document Number:  2021-111P \n',
  'bbox': (40.56, 639.88896, 204.6696, 650.92896),
  'fontname': None,
  'fontsize': None,
  'page_num': 1,
  'type': pdfminer.layout.LTTextBoxHorizontal,
  'direction': None,
  'bbox.x0': 40.56,
  'bbox.y0': 639.88896,
  'bbox.x1': 204.6696,
  'bbox.y1': 650.92896},
 {'text': 'Document Title: \n',
  'bbox': (40.56000000000003, 614.69568, 126.79344000000003, 625.73568),
  'fontname': None,
  'fontsize': None,
  'page_num': 1,
  'type': pdfminer.layout.LTTextBoxHorizontal,
  'direction': None,
  'bbox.x0': 40.56000000000003,
  'bbox.y0': 614.69568,
  'bbox.x1': 126.79344000000003,
  'bbox.y1': 625.73568},
 {'text': 'BLOCK STUDY AND PRELIMINARY DESIGN – BLOCK

In [6]:
import autoextractpdf
import pandas as pd
import numpy as np
import math as mt

df_pdf_data = pd.DataFrame(extracted_text)

autoextractpdf.process_pdf_auto(df_pdf_data=df_pdf_data,
                                filename=my_file,
                                extract_by_page=True,
                                output_path=base_path + "out/",
                                extract_images=False,
                                filecontent=None,
                                direction="dataframe",
                                save_text_coordinates_to_local=False,
                                save_file_to_local=True)



{'source_filename': '../in/2021-111P.pdf',
 'process_date': '2024-12-17',
 'total_pages': '84',
 'report_type': 'Auto-Extraction',
 'report_content': [{'page_num': 1,
   'section_id': 0,
   'section_name': 'Page-1_group-0',
   'direction': 'text',
   'data_list': ' Request for Proposal D ocument Number:  2021-111P D ocument Title: BLOCK STUDY AND PRELIMINARY DESIGN – BLOCK 1 WITHIN THE CITY OF  MISSISSAUGA D ate Issued: Monday, February 22, 2021 E LECTRONIC BID SUBMISSIONS ONLY shall be received by the Agency through the Bidding System no later than:   12:00 noon local time  Wednesday, March 17, 2021 I t is the Bidder’s sole responsibility to ensure that:  • the submission is received electronically by the Agency through the Bidding System by the  date and time specified above  • the submission is accompanied by all required documentation P rocurement Representative: Amanda Tanti, Procurement Analyst Telephone Number: (905) 791-7800, ext. 4165  '},
  {'page_num': 2,
   'section_id': 1,